# 2. Data Preprocessing

*Proposal section 3.2.2.* Cleans the raw transactions: parses amounts, flags failed transactions, encodes the payment channel, and attaches merchant category names. Saves the cleaned table for `03_feature_engineering`.

## Setup

Runs locally or in Google Colab. In Colab it mounts Google Drive, copies this repo's code from GitHub, and points `data/` at **My Drive** (raw files in the Drive root, outputs in `My Drive/processed`) and `models/` at `My Drive/models`, so outputs survive between sessions.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_URL = "https://github.com/JamieNguru/169421-escrow-based-wallet-and-payment-system.git"
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "dev", REPO_URL, "/content/repo"], check=True)
    ROOT = Path("/content/repo/trust-scoring-ml")
    DRIVE = Path("/content/drive/MyDrive")
    (DRIVE / "models").mkdir(exist_ok=True)
    for name, target in [("data", DRIVE), ("models", DRIVE / "models")]:
        if not (ROOT / name).exists():
            (ROOT / name).symlink_to(target, target_is_directory=True)
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from src.preprocessing import (
    clean_amount,
    encode_use_chip,
    flag_errors,
    load_transactions,
    merge_mcc_descriptions,
)

## Load

Only the columns the pipeline uses are loaded, with compact types (e.g. 16-bit user IDs), which keeps the 13.3M rows within a laptop's memory. No rows or values are dropped.

In [3]:
transactions = load_transactions(DATA_DIR / "transactions_data.csv")
print(f"{len(transactions):,} rows")
transactions.dtypes

13,305,915 rows


id                      int32
date           datetime64[us]
client_id               int16
card_id                 int16
amount                    str
use_chip             category
merchant_id             int32
mcc                     int16
errors                    str
dtype: object

## Clean the amount column

Amounts are stored as text such as `$-77.00`. Negative amounts are refunds or reversals and are kept.

In [4]:
print("Raw:", transactions["amount"].head(3).tolist())
transactions = clean_amount(transactions)
print("Missing amounts after cleaning:", transactions["amount"].isna().sum())
transactions["amount"].describe().round(2)

Raw: ['$-77.00', '$14.57', '$80.00']


Missing amounts after cleaning: 0


count    13305915.00
mean           42.98
std            81.66
min          -500.00
25%             8.93
50%            28.99
75%            63.71
max          6820.20
Name: amount, dtype: float64

## Flag failed transactions

`has_error` marks any transaction with an error (bad PIN, insufficient balance, technical glitch, ...). It becomes the basis of the dispute and refund features.

In [5]:
transactions = flag_errors(transactions)
print(f"Failed transactions: {transactions['has_error'].sum():,} ({transactions['has_error'].mean():.2%})")

Failed transactions: 211,393 (1.59%)


## Encode the payment channel

In [6]:
transactions = encode_use_chip(transactions)
channel_columns = [c for c in transactions.columns if c.startswith("channel_")]
transactions[channel_columns].sum()

channel_Online Transaction    1557912
channel_Swipe Transaction     6967185
channel_Chip Transaction      4780818
dtype: int64

## Attach merchant category names

In [7]:
transactions = merge_mcc_descriptions(transactions, DATA_DIR / "mcc_codes.json")
print("Transactions with no matching category:", transactions["mcc_description"].isna().sum())
transactions["mcc_description"].value_counts().head(10)

Transactions with no matching category: 0


mcc_description
Grocery Stores, Supermarkets     1592584
Miscellaneous Food Stores        1460875
Service Stations                 1424711
Eating Places and Restaurants     999738
Drug Stores and Pharmacies        772913
Tolls and Bridge Fees             674135
Wholesale Clubs                   601942
Money Transfer                    589140
Taxicabs and Limousines           500662
Fast Food Restaurants             499659
Name: count, dtype: int64

## Save

In [8]:
output_path = PROCESSED_DIR / "transactions_clean.parquet"
transactions.to_parquet(output_path, index=False)
print(f"Saved {len(transactions):,} rows x {transactions.shape[1]} columns to "
      f"{output_path.relative_to(ROOT).as_posix()} ({output_path.stat().st_size / 1e6:.0f} MB)")

Saved 13,305,915 rows x 13 columns to data/processed/transactions_clean.parquet (221 MB)
